# House Price Prediction - Kaggle Submission

Clean, structured pipeline with preprocessing, training, and submission.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [ ]:
# Load dataset
import kagglehub
path = kagglehub.dataset_download("ericmauricio/home-data-for-ml-course")

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))

train.head()


In [ ]:
# Save IDs
train_id = train["Id"]
test_id = test["Id"]

# Split features and target
X = train.drop(["SalePrice", "Id"], axis=1)
y = train["SalePrice"]

X_test = test.drop(["Id"], axis=1)

# Column types
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

print("Categorical:", len(categorical_cols))
print("Numerical:", len(numerical_cols))


In [ ]:
# Preprocessing pipelines

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])


In [ ]:
# Model pipeline

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        random_state=42
    ))
])


In [ ]:
# Train model
model.fit(X, y)

In [ ]:
# Predict on test set
predictions = model.predict(X_test)


In [ ]:
# Create submission file

submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()